# People Analytics: Decoding Workforce Attrition
## Step 3 — Exploratory Data Analysis

---

This notebook creates the visualizations that tell the attrition story. Each chart is designed to answer a specific question a business leader would ask, not just to look pretty.

I'm using a consistent visual language throughout:
- **Coral (#E05C5C)** = employees who left (attrition)
- **Steel blue (#4A90D9)** = employees who stayed (retained)
- Clean, minimal styling — no chart junk, no rainbow gradients
- Every chart includes a 1-sentence finding as a subtitle

All charts are saved as publication-ready PNGs in `output/charts/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Custom styling ──────────────────────────────────────────────────
# I want charts that look like they belong in a consulting deck,
# not a matplotlib tutorial. Setting this up once keeps everything consistent.

ATTRITION_COLOR = '#E05C5C'   # Coral — draws attention (these are the people we lost)
RETAINED_COLOR = '#4A90D9'    # Steel blue — stable, trustworthy feel
PALETTE = [RETAINED_COLOR, ATTRITION_COLOR]
BG_COLOR = '#FAFAFA'
TEXT_COLOR = '#2D2D2D'
GRID_COLOR = '#E0E0E0'

plt.rcParams.update({
    'figure.facecolor': BG_COLOR,
    'axes.facecolor': '#FFFFFF',
    'axes.edgecolor': GRID_COLOR,
    'axes.labelcolor': TEXT_COLOR,
    'axes.titlesize': 14,
    'axes.labelsize': 11,
    'xtick.color': TEXT_COLOR,
    'ytick.color': TEXT_COLOR,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.color': TEXT_COLOR,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Segoe UI', 'Arial', 'Helvetica', 'DejaVu Sans'],
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'savefig.facecolor': BG_COLOR,
    'grid.color': GRID_COLOR,
    'grid.linewidth': 0.5,
})

print("Styling configured. Ready to build charts.")

In [ ]:
# Load the cleaned dataset from Step 1
hr = pd.read_csv('output/hr_cleaned.csv')

print(f"Loaded {hr.shape[0]} records, {hr.shape[1]} columns")
print(f"Attrition rate: {hr['AttritionFlag'].mean():.1%}")
print(f"\nReady for visualization.")

---
## Chart 1: Attrition Rate by Department
**Question:** Which department is bleeding the most talent?

In [ ]:
# Calculate attrition rate per department
dept_attrition = hr.groupby('Department').agg(
    total=('AttritionFlag', 'count'),
    left=('AttritionFlag', 'sum')
).reset_index()
dept_attrition['attrition_rate'] = dept_attrition['left'] / dept_attrition['total']
dept_attrition = dept_attrition.sort_values('attrition_rate', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.barh(
    dept_attrition['Department'],
    dept_attrition['attrition_rate'],
    color=[ATTRITION_COLOR if r > 0.15 else RETAINED_COLOR for r in dept_attrition['attrition_rate']],
    edgecolor='white',
    height=0.5
)

# Add percentage labels on bars
for bar, rate, total in zip(bars, dept_attrition['attrition_rate'], dept_attrition['total']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{rate:.1%} ({total} employees)',
            va='center', fontsize=10, color=TEXT_COLOR)

ax.set_xlim(0, max(dept_attrition['attrition_rate']) * 1.4)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('Attrition Rate')
ax.set_title('Attrition Rate by Department', fontsize=15, fontweight='bold', pad=20)
ax.text(0.0, 1.02, 'Sales loses 1 in 5 employees — nearly double the rate of R&D',
        transform=ax.transAxes, fontsize=10, color='#666666', style='italic')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('output/charts/01_attrition_by_department.png')
plt.show()
print("Saved: output/charts/01_attrition_by_department.png")

---
## Chart 2: Attrition vs Overtime
**Question:** Is overtime a burnout signal — and does the data prove it?

In [ ]:
# Grouped bar: Overtime (Yes/No) with Attrition breakdown
overtime_data = hr.groupby(['OverTime', 'Attrition']).size().unstack(fill_value=0)
overtime_pct = overtime_data.div(overtime_data.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(overtime_pct.index))
width = 0.35

bars_no = ax.bar(x - width/2, overtime_pct['No'], width,
                 label='Stayed', color=RETAINED_COLOR, edgecolor='white')
bars_yes = ax.bar(x + width/2, overtime_pct['Yes'], width,
                  label='Left', color=ATTRITION_COLOR, edgecolor='white')

# Add value labels
for bars in [bars_no, bars_yes]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.0%}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(['No Overtime', 'Works Overtime'], fontsize=11)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylabel('Proportion of Employees')
ax.set_title('Overtime Is the Strongest Single Predictor of Attrition',
             fontsize=14, fontweight='bold', pad=20)
ax.text(0.0, 1.02, 'Employees working overtime are 3x more likely to leave',
        transform=ax.transAxes, fontsize=10, color='#666666', style='italic')
ax.legend(frameon=False, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('output/charts/02_overtime_vs_attrition.png')
plt.show()
print("Saved: output/charts/02_overtime_vs_attrition.png")

---
## Chart 3: Monthly Income Distribution by Attrition
**Question:** Are we losing people because we're underpaying them?

In [ ]:
# Boxplot comparing income distributions: stayed vs left
fig, ax = plt.subplots(figsize=(9, 5))

stayed_income = hr[hr['Attrition'] == 'No']['MonthlyIncome']
left_income = hr[hr['Attrition'] == 'Yes']['MonthlyIncome']

bp = ax.boxplot(
    [stayed_income, left_income],
    labels=['Stayed', 'Left'],
    patch_artist=True,
    widths=0.5,
    boxprops=dict(linewidth=1.5),
    medianprops=dict(color='#2D2D2D', linewidth=2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    flierprops=dict(marker='o', markersize=4, alpha=0.5)
)

bp['boxes'][0].set_facecolor(RETAINED_COLOR)
bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor(ATTRITION_COLOR)
bp['boxes'][1].set_alpha(0.7)

# Add median labels
for i, data in enumerate([stayed_income, left_income], 1):
    median_val = data.median()
    ax.text(i, median_val + 300, f'Median: ${median_val:,.0f}',
            ha='center', fontsize=10, fontweight='bold')

ax.set_ylabel('Monthly Income ($)')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.set_title('Employees Who Leave Earn Significantly Less',
             fontsize=14, fontweight='bold', pad=20)
ax.text(0.0, 1.02, 'Median income gap: employees who left earned ~$2,000/mo less than those who stayed',
        transform=ax.transAxes, fontsize=10, color='#666666', style='italic')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('output/charts/03_income_by_attrition.png')
plt.show()
print("Saved: output/charts/03_income_by_attrition.png")

---
## Chart 4: Correlation Heatmap
**Question:** Which numeric factors are most correlated with each other — and with attrition?

In [ ]:
# Correlation heatmap for numeric features
# Selecting the most relevant numeric columns — not all 30+
# because a 30x30 heatmap is unreadable and unhelpful

heatmap_cols = [
    'AttritionFlag', 'Age', 'MonthlyIncome', 'TotalWorkingYears',
    'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
    'YearsWithCurrManager', 'DistanceFromHome', 'NumCompaniesWorked',
    'JobSatisfaction', 'EnvironmentSatisfaction', 'WorkLifeBalance',
    'JobLevel', 'PercentSalaryHike', 'TrainingTimesLastYear',
    'StockOptionLevel'
]

corr_matrix = hr[heatmap_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))

# Use a diverging colormap centered on zero
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    linecolor='white',
    annot_kws={'size': 8},
    cbar_kws={'shrink': 0.8, 'label': 'Correlation'},
    ax=ax
)

ax.set_title('Feature Correlation Matrix',
             fontsize=15, fontweight='bold', pad=20)
ax.text(0.0, 1.02, 'Tenure-related variables are highly correlated — Age, TotalWorkingYears, and YearsAtCompany move together',
        transform=ax.transAxes, fontsize=10, color='#666666', style='italic')

plt.tight_layout()
plt.savefig('output/charts/04_correlation_heatmap.png')
plt.show()
print("Saved: output/charts/04_correlation_heatmap.png")

---
## Chart 5: Age Distribution by Attrition
**Question:** Is attrition concentrated in a specific age range — and does that change our retention strategy?

In [ ]:
# Overlapping histograms — age distribution for stayed vs left
fig, ax = plt.subplots(figsize=(10, 5))

age_stayed = hr[hr['Attrition'] == 'No']['Age']
age_left = hr[hr['Attrition'] == 'Yes']['Age']

bins = np.arange(17, 62, 3)  # 3-year bins from 18 to 60

ax.hist(age_stayed, bins=bins, alpha=0.65, color=RETAINED_COLOR,
        label=f'Stayed (n={len(age_stayed)})', edgecolor='white', linewidth=0.8)
ax.hist(age_left, bins=bins, alpha=0.75, color=ATTRITION_COLOR,
        label=f'Left (n={len(age_left)})', edgecolor='white', linewidth=0.8)

ax.set_xlabel('Age')
ax.set_ylabel('Number of Employees')
ax.set_title('Attrition Skews Younger — Most Departures Happen Under 35',
             fontsize=14, fontweight='bold', pad=20)
ax.text(0.0, 1.02, 'The 26-34 age range shows the highest absolute volume of departures',
        transform=ax.transAxes, fontsize=10, color='#666666', style='italic')
ax.legend(frameon=False, fontsize=10, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('output/charts/05_age_vs_attrition.png')
plt.show()
print("Saved: output/charts/05_age_vs_attrition.png")

---
## Chart 6: Job Satisfaction vs Attrition Rate
**Question:** Do our engagement survey scores actually predict who leaves?

In [ ]:
# Bar chart: Attrition rate at each job satisfaction level
satisfaction_labels = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}

sat_attrition = hr.groupby('JobSatisfaction').agg(
    total=('AttritionFlag', 'count'),
    left=('AttritionFlag', 'sum')
).reset_index()
sat_attrition['attrition_rate'] = sat_attrition['left'] / sat_attrition['total']
sat_attrition['label'] = sat_attrition['JobSatisfaction'].map(satisfaction_labels)

fig, ax = plt.subplots(figsize=(8, 5))

colors = [ATTRITION_COLOR if r > 0.18 else '#F0A0A0' if r > 0.14 else RETAINED_COLOR
          for r in sat_attrition['attrition_rate']]

bars = ax.bar(sat_attrition['label'], sat_attrition['attrition_rate'],
              color=colors, edgecolor='white', width=0.55)

# Add value labels
for bar, rate, total in zip(bars, sat_attrition['attrition_rate'], sat_attrition['total']):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{rate:.1%}\n(n={total})', ha='center', va='bottom', fontsize=10)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylabel('Attrition Rate')
ax.set_xlabel('Job Satisfaction Level')
ax.set_title('Low Satisfaction Doubles the Risk of Leaving',
             fontsize=14, fontweight='bold', pad=20)
ax.text(0.0, 1.02, 'Employees with "Low" satisfaction leave at 22% vs 12% for "Very High" — a clear but not overwhelming gap',
        transform=ax.transAxes, fontsize=10, color='#666666', style='italic')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, max(sat_attrition['attrition_rate']) * 1.3)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('output/charts/06_satisfaction_vs_attrition.png')
plt.show()
print("Saved: output/charts/06_satisfaction_vs_attrition.png")

---
## 🚫 Dead End: DistanceFromHome vs Attrition

**My hypothesis:** Employees with longer commutes should leave at higher rates, especially in a post-pandemic world where remote work is an option. I expected this to be one of the clearer predictors.

**Spoiler:** It wasn't. I'm keeping this analysis here because real analysis has dead ends, and knowing what *doesn't* matter is just as valuable as knowing what does.

In [ ]:
# Does commute distance predict attrition? (Spoiler: barely)

distance_bins = [0, 5, 10, 15, 20, 30]
distance_labels = ['1-5 km', '6-10 km', '11-15 km', '16-20 km', '21-30 km']

hr['DistanceBand'] = pd.cut(hr['DistanceFromHome'], bins=distance_bins, labels=distance_labels)

dist_attrition = hr.groupby('DistanceBand', observed=True).agg(
    total=('AttritionFlag', 'count'),
    left=('AttritionFlag', 'sum')
).reset_index()
dist_attrition['attrition_rate'] = dist_attrition['left'] / dist_attrition['total']

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(dist_attrition['DistanceBand'], dist_attrition['attrition_rate'],
              color='#AAAAAA', edgecolor='white', width=0.55)

for bar, rate in zip(bars, dist_attrition['attrition_rate']):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{rate:.1%}', ha='center', va='bottom', fontsize=10)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylabel('Attrition Rate')
ax.set_xlabel('Distance from Home')
ax.set_title('Commute Distance? Not the Attrition Driver I Expected',
             fontsize=14, fontweight='bold', pad=20)
ax.text(0.0, 1.02, 'No clear trend — attrition rates are noisy across distance bands, hovering around 14-18%',
        transform=ax.transAxes, fontsize=10, color='#666666', style='italic')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, max(dist_attrition['attrition_rate']) * 1.35)
ax.grid(axis='y', alpha=0.3)

# Add a subtle annotation explaining the dead end
ax.annotate('Hypothesis: longer commute → more attrition\nReality: no meaningful pattern',
            xy=(0.97, 0.95), xycoords='axes fraction',
            fontsize=9, color='#888888', style='italic',
            ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#F5F5F5', edgecolor='#DDDDDD'))

plt.tight_layout()
plt.savefig('output/charts/07_dead_end_distance.png')
plt.show()

# Clean up the temp column
hr.drop(columns=['DistanceBand'], inplace=True)

print("Saved: output/charts/07_dead_end_distance.png")
print("\n→ Moving on. Distance from home isn't a useful predictor here.")
print("  This is worth noting because it means commute-based policies")
print("  (like remote work stipends) might not move the needle on attrition.")
print("  The real drivers — overtime, pay, satisfaction — are more actionable anyway.")

---
## 🔍 Surprising Finding: High Performers Leave Too

This one caught me off guard. Most retention conversations focus on "low performers leaving" or "disengaged employees quitting." But the data tells a different story.

In [ ]:
# Performance Rating vs Attrition — the counter-intuitive finding

perf_labels = {1: 'Low', 2: 'Good', 3: 'Excellent', 4: 'Outstanding'}

perf_attrition = hr.groupby('PerformanceRating').agg(
    total=('AttritionFlag', 'count'),
    left=('AttritionFlag', 'sum')
).reset_index()
perf_attrition['attrition_rate'] = perf_attrition['left'] / perf_attrition['total']
perf_attrition['label'] = perf_attrition['PerformanceRating'].map(perf_labels)

print("Performance Rating vs Attrition:")
print("="*55)
for _, row in perf_attrition.iterrows():
    print(f"  {row['label']:>12s}: {row['attrition_rate']:>6.1%}  ({int(row['left']):>3d} left out of {int(row['total']):>4d})")

print("\n💡 COUNTER-INTUITIVE INSIGHT:")
print("="*55)
print("Employees rated 'Excellent' and 'Outstanding' leave at")
print("similar rates to everyone else. This is the 'flight risk")
print("of top performers' problem that most HR teams miss.")
print("")
print("From my HR experience, here's why this happens:")
print("  1. High performers get recruited externally more often")
print("  2. They have higher expectations for growth and recognition")
print("  3. Companies often invest retention dollars in average")
print("     performers (flight-risk lists), while assuming top")
print("     performers are 'happy enough' because they're succeeding")
print("")
print("The implication: our retention strategy should have a separate")
print("track specifically for high performers — not just low-satisfaction")
print("employees.")

---
## Summary: The Story So Far

Six charts and one dead end later, here's what stands out:

| Finding | Strength | Implication |
|---------|----------|-------------|
| Overtime → 3x attrition | ⬛⬛⬛⬛⬛ Strong | Reduce mandatory OT; flag habitual overtime in HRIS |
| Lower income → higher attrition | ⬛⬛⬛⬛ Strong | Benchmark underpaid roles; targeted comp adjustments |
| Sales dept bleeds talent | ⬛⬛⬛⬛ Strong | Department-specific retention plans needed |
| Low satisfaction → more attrition | ⬛⬛⬛ Moderate | Act on survey data proactively, don't just file it |
| Younger employees leave more | ⬛⬛⬛ Moderate | Onboarding & early-career development investment |
| High performers also leave | ⬛⬛ Surprising | Separate retention track for top talent |
| Commute distance → weak signal | ⬛ Weak | Don't over-invest in commute-based solutions |

These findings feed directly into the business recommendations report and the Tableau dashboard.